# 06 — Data Science Dashboards: Design, Status & Accessibility

> **📓 Notebook · Module 03 · Beginner**  
> *A Data Science application is not simply a notebook placed inside a web page.*

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Explain the difference between a Jupyter notebook and a Streamlit dashboard.
2. Apply progressive disclosure: KPIs → Charts → Details → Raw Data.
3. Use `st.metric` effectively for KPI communication.
4. Implement loading, progress, and status feedback with `st.spinner`, `st.progress`, `st.status`, `st.toast`.
5. Apply basic accessibility principles (contrast, labels, alt text, heading hierarchy).
6. Avoid common UI complexity traps.
7. Build a complete Data Science dashboard from scratch.

## 📋 Prerequisites

- Completed [Notebook 05 — Layouts & Containers](05_layouts_and_containers.ipynb)
- Understanding of sidebar, columns, tabs, expanders
- Pandas DataFrames, basic plotting

---

## 📚 Concept: Notebook vs. Dashboard

A Jupyter notebook is a **personal analysis tool**. A Streamlit dashboard is a **shared insight tool**.

```
NOTEBOOK (Analyst)              DASHBOARD (Audience)
┌─────────────────┐            ┌──────────────────────┐
│ Cell 1: import  │            │ KPIs: Revenue, Users │
│ Cell 2: load df │            │ [Chart] [Table]      │
│ Cell 3: explore │    ═══>    │ [Details expand]     │
│ Cell 4: plot    │            │ Filters: sidebar     │
│ Cell 5: stats   │            │ Export: download      │
│ Cell 6: insights│            │                      │
└─────────────────┘            └──────────────────────┘
  Shows PROCESS                  Shows RESULTS
  Requires CODE knowledge        Requires DOMAIN knowledge
  One format for all             Adapts to user choices
```

## 🧠 Intuition: The 5-Second Rule

A stakeholder should understand your dashboard within **5 seconds**:

1. **What is this about?** → Title + KPI row
2. **Is it good or bad?** → Metric deltas (green/red arrows)
3. **What's the trend?** → Primary chart
4. **Can I explore?** → Sidebar controls + tabs
5. **Can I get the data?** → Export buttons

If any of these are missing, the dashboard fails the 5-second test.

---

## 🔧 Build It: KPI Metrics

`st.metric` is the primary tool for communicating key performance indicators.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Notebook 06", layout="wide")

# Generate sample sales data
np.random.seed(42)
dates = pd.date_range("2026-01-01", periods=90, freq="D")
sales_df = pd.DataFrame({
    "Date": dates,
    "Revenue": np.random.randint(25000, 65000, 90),
    "Users": np.random.randint(800, 2500, 90),
    "Orders": np.random.randint(50, 300, 90),
    "Returns": np.random.randint(0, 20, 90)
})
sales_df["AvgOrder"] = (sales_df["Revenue"] / sales_df["Orders"]).round(2)
sales_df["Conversion"] = (sales_df["Orders"] / sales_df["Users"] * 100).round(1)

st.header("KPI Metrics — The Dashboard Header")

total_revenue = sales_df["Revenue"].sum()
total_users = sales_df["Users"].sum()
total_orders = sales_df["Orders"].sum()
avg_conversion = sales_df["Conversion"].mean()

k1, k2, k3, k4 = st.columns(4)
k1.metric("Total Revenue", f"${total_revenue:,.0f}", "+8.3%", icon="💰")
k2.metric("Total Users", f"{total_users:,}", "+5.1%", icon="👥")
k3.metric("Total Orders", f"{total_orders:,}", "+3.7%", icon="📦")
k4.metric("Avg Conversion", f"{avg_conversion:.1f}%", "+0.4%", icon="🎯")

### Metric Design Rules

```python
# ✅ 4 metrics per row — the standard
c1, c2, c3, c4 = st.columns(4)

# ✅ Short labels
c1.metric("Revenue", "$1.2M", "+8.3%")

# ✅ Delta tells the story
# Green = positive, Red = negative (by default)

# ✅ Format large numbers
"$1,234,567" → "$1.2M"
"12,345" → "12.3K"

# ✅ Icons add context
st.metric("Revenue", "$1.2M", icon="💰")
```

### When Lower Is Better

Some metrics (error rates, bounce rates) should decrease. Use `delta_color="inverse"` to flip the colors.

In [ ]:
st.subheader("Inverse Delta Colors")

col1, col2 = st.columns(2)
with col1:
    # Normal: green = good (revenue went up)
    st.metric("Revenue", "$1.2M", "+8.3%", delta_color="normal", icon="💰")
with col2:
    # Inverse: green = good (error rate went DOWN)
    st.metric("Error Rate", "0.3%", "-0.1%", delta_color="inverse", icon="🔧")

---

## 🔧 Build It: Status & Progress Feedback

Users need to know **what's happening** while your app processes data.

In [ ]:
import time

st.header("Progress & Status Elements")

# 1. Progress Bar
st.subheader("Progress Bar")
progress = st.progress(0, text="Starting...")
for i in range(100):
    time.sleep(0.01)
    progress.progress(i + 1, text=f"Processing... {i + 1}%")
progress.empty()  # Clean up
st.success("Processing complete!")

In [ ]:
# 2. Spinner
st.subheader("Spinner (Quick Tasks)")

with st.spinner("Loading data..."):
    time.sleep(1)  # Simulate quick load

st.write("Data loaded!")

In [ ]:
# 3. Status Container
st.subheader("Status Container (Multi-Step Pipeline)")

with st.status("Running analysis pipeline...", expanded=True) as status:
    st.write("📥 Step 1: Loading raw data...")
    time.sleep(0.5)

    st.write("🧹 Step 2: Cleaning and transforming...")
    time.sleep(0.5)

    st.write("📊 Step 3: Computing metrics...")
    time.sleep(0.5)

    st.write("✅ Step 4: Generating report...")
    time.sleep(0.5)

    status.update(label="Pipeline complete!", state="complete")

In [ ]:
# 4. Toast Notifications
st.subheader("Toast Notifications")

if st.button("Show Toast"):
    st.toast("Data refreshed!", icon="✅")

In [ ]:
# 5. Callout Messages
st.subheader("Callout Messages")

col1, col2 = st.columns(2)
with col1:
    st.success("✅ Report generated successfully!")
    st.info("ℹ️ Last updated: 2 minutes ago")
with col2:
    st.warning("⚠️ Data is 30 days old")
    st.error("❌ API connection failed")

### Status Feedback Decision Guide

| Situation | Element | Duration |
|---|---|---|
| Loading data (known %) | `st.progress()` | 1-30 seconds |
| Quick computation | `st.spinner()` | 0.5-5 seconds |
| Multi-step pipeline | `st.status()` | 5-60 seconds |
| Action confirmed | `st.toast()` | 2-3 seconds |
| Error occurred | `st.error()` | Persistent |
| Task completed | `st.success()` | Persistent |

---

## 🧪 Experiment: Building a Complete Dashboard

Let's apply everything to build a **Sales Analytics Dashboard**.

In [ ]:
st.header("🧪 Sales Analytics Dashboard")
st.caption("A Data Science application — not a notebook in a browser.")

# --- Sidebar Controls ---
with st.sidebar:
    st.header("🔍 Filters")
    date_range = st.date_input(
        "Date range",
        value=(sales_df["Date"].min(), sales_df["Date"].max())
    )
    metric_view = st.selectbox("Primary metric", ["Revenue", "Users", "Orders"])
    show_returns = st.checkbox("Include returns analysis")

    st.header("⚙️ Display")
    chart_type = st.radio("Chart type", ["Line", "Bar", "Area"], horizontal=True)

# --- Filter data ---
if isinstance(date_range, tuple) and len(date_range) == 2:
    mask = (sales_df["Date"] >= pd.Timestamp(date_range[0])) & \
           (sales_df["Date"] <= pd.Timestamp(date_range[1]))
    filtered = sales_df[mask]
else:
    filtered = sales_df

# --- KPI Row ---
total_rev = filtered["Revenue"].sum()
total_users = filtered["Users"].sum()
total_orders = filtered["Orders"].sum()
avg_conv = filtered["Conversion"].mean()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Revenue", f"${total_rev:,.0f}", icon="💰")
c2.metric("Users", f"{total_users:,}", icon="👥")
c3.metric("Orders", f"{total_orders:,}", icon="📦")
c4.metric("Avg Conversion", f"{avg_conv:.1f}%", icon="🎯")

# --- Main Content: Tabs ---
tab_trend, tab_breakdown, tab_details = st.tabs(["📈 Trends", "📊 Breakdown", "📋 Details"])

with tab_trend:
    chart_col, summary_col = st.columns([2, 1])
    with chart_col:
        chart_data = filtered.set_index("Date")[[metric_view]]
        if chart_type == "Line":
            st.line_chart(chart_data)
        elif chart_type == "Bar":
            st.bar_chart(chart_data)
        else:
            st.area_chart(chart_data)
    with summary_col:
        st.subheader("Summary")
        st.write(f"**Period:** {len(filtered)} days")
        st.write(f"**Avg Daily {metric_view}:** {filtered[metric_view].mean():,.0f}")
        st.write(f"**Peak Day:** {filtered.loc[filtered[metric_view].idxmax(), 'Date'].strftime('%b %d')}")

with tab_breakdown:
    if show_returns:
        col1, col2 = st.columns(2)
        with col1:
            st.bar_chart(filtered.set_index("Date")[["Revenue"]])
        with col2:
            st.bar_chart(filtered.set_index("Date")[["Returns"]])
    else:
        st.bar_chart(filtered.set_index("Date")[["Revenue", "Users", "Orders"]])

with tab_details:
    st.dataframe(filtered, use_container_width=True, hide_index=True)
    with st.expander("📥 Export Data"):
        csv = filtered.to_csv(index=False)
        st.download_button("Download CSV", csv, "sales_data.csv")

---

## 🧪 Experiment: Accessibility in Practice

Let's apply accessibility principles to our dashboard.

In [ ]:
st.header("Accessibility Checklist")

checklist = [
    ("✅", "Color is not the only signal", "Use icons + text + color together"),
    ("✅", "Alt text on images", "st.image(..., alt_text='Description')"),
    ("✅", "Logical heading hierarchy", "title → header → subheader (h1 → h2 → h3)"),
    ("✅", "Widget labels are descriptive", "'Product category' not 'Select one'"),
    ("✅", "Sufficient contrast", "Streamlit default theme meets WCAG AA"),
    ("✅", "Keyboard navigation", "All widgets are keyboard-navigable by default"),
]

for icon, principle, detail in checklist:
    st.write(f"{icon} **{principle}** — {detail}")

st.divider()

# Example: Good vs Bad metric accessibility
st.subheader("Good vs Bad: Metric Accessibility")

col1, col2 = st.columns(2)

with col1:
    st.markdown("**❌ Bad: Color only**")
    st.metric("Status", "OK", "+5%", delta_color="normal")
    # If user can't see colors, they miss the signal

with col2:
    st.markdown("**✅ Good: Color + Icon + Text**")
    st.metric("Status", "OK", "+5%", icon="✅", delta_color="normal")
    # Icon and text provide the signal; color reinforces it

---

## 🧪 Experiment: Avoiding UI Complexity

Compare a cluttered layout vs. a clean one.

In [ ]:
st.header("Complexity Comparison")

col_bad, col_good = st.columns(2)

with col_bad:
    st.markdown("**❌ Overly Complex**")
    st.write("- 8 columns of metrics")
    st.write("- 6 charts on one page")
    st.write("- Nested columns 3 levels deep")
    st.write("- 12 sidebar widgets")
    st.write("- No visual hierarchy")
    st.write("- All data shown at once")

with col_good:
    st.markdown("**✅ Clean & Focused**")
    st.write("- 4 KPI metrics at top")
    st.write("- 1 primary chart per tab")
    st.write("- Flat column structure")
    st.write("- Grouped sidebar controls")
    st.write("- Clear visual hierarchy")
    st.write("- Progressive disclosure")

---

## ⚠️ Common Mistakes

| Mistake | Why It's Bad | Fix |
|---|---|---|
| Showing raw data first | Users must interpret without guidance | Lead with KPIs and insights |
| 8+ columns in one row | Unreadable on any screen | Max 4 metric columns |
| No loading feedback | Users think app is frozen | Use `st.spinner()` or `st.progress()` |
| Color-only status signals | Inaccessible to colorblind users | Add icons and text |
| Moving elements between reruns | Breaks user's spatial memory | Keep layout consistent |
| Deep column nesting | Hard to read and maintain | Use tabs or expanders instead |
| Showing everything at once | Visual overload | Progressive disclosure |
| Missing titles/headers | Users don't know what they're seeing | Always start with `st.title()` |

---

## 🔍 Debugging Tips

| Symptom | Likely Cause | Fix |
|---|---|---|
| Metrics show raw numbers | Missing `format` or large values | Use `f"${val:,.0f}"` formatting |
| Spinner doesn't appear | Task too fast | Add `time.sleep(0.5)` for demo |
| Progress bar stuck at 0 | Wrong loop range | Use `range(100)` and `progress(i+1)` |
| Toast not visible | Browser blocking notifications | Check browser notification settings |
| Layout shifts on rerun | Inconsistent widget rendering | Ensure widgets always render (no conditional widgets) |
| Sidebar overlaps main | `layout="centered"` on narrow screens | Use `layout="wide"` for dashboards |

---

## ✅ Best Practices

1. **Lead with insights** — KPIs first, charts second, data third.
2. **Progressive disclosure** — let users drill down, don't overwhelm.
3. **Use status feedback** — spinners for quick tasks, progress bars for measurable ones.
4. **4 metrics per row** — the standard for KPI displays.
5. **Accessibility from day one** — contrast, labels, alt text, keyboard nav.
6. **Consistent layout** — users build spatial memory.
7. **Minimal cognitive load** — every element must earn its place.
8. **The 5-second test** — can a user understand the app in 5 seconds?

---

## ✏️ Exercises

### Exercise 1: KPI Dashboard
Build a dashboard with:
- 4 `st.metric` cards with appropriate icons and deltas
- A 70/30 column split (chart left, summary right)
- An expander for raw data

### Exercise 2: Status Feedback
Create an app that:
- Shows a spinner while "loading" data
- Shows a progress bar during "processing"
- Shows a status container for a 3-step pipeline
- Shows a toast on completion

### Exercise 3: Accessibility Audit
Take any dashboard and:
1. Add icons to all metrics
2. Add alt_text to all images
3. Ensure heading hierarchy is correct
4. Add `help` parameters to all widgets
5. Check that color isn't the only signal

## 🚀 Challenge Problem

Build a **complete Sales Analytics Dashboard** that includes:

1. **Sidebar** with form-based filters (date range, metric selection, chart type)
2. **KPI row** — 4 metrics with icons, deltas, and proper formatting
3. **Status pipeline** — show progress when "loading" data
4. **Tabs** — Trends, Breakdown, Details
5. **70/30 column layout** in the Trends tab
6. **Progressive disclosure** — expanders for raw data and export
7. **Accessibility** — alt text, heading hierarchy, descriptive labels
8. **Toast notifications** on user actions

Use synthetic data. The dashboard should pass the 5-second test.

---

## 📌 Key Takeaways

1. **A dashboard is not a notebook in a browser** — it shows results, not process.
2. **Lead with KPIs** — stakeholders need answers in 5 seconds.
3. **Progressive disclosure** — KPIs → Charts → Details → Raw Data.
4. **Status feedback** communicates progress — spinners, progress bars, status containers.
5. **Accessibility** means contrast, labels, alt text, and keyboard navigation.
6. **Simplicity is a feature** — every element must earn its place.
7. **Consistent layout** builds spatial memory.

---

## 📚 Further Reading

- [Streamlit Metrics API](https://docs.streamlit.io/develop/api-reference/data/st.metric)
- [Streamlit Status Elements](https://docs.streamlit.io/develop/api-reference/status)
- [Streamlit Theming](https://docs.streamlit.io/develop/concepts/theming)
- [WCAG 2.1 Quick Reference](https://www.w3.org/WAI/WCAG21/quickref/)

---

## 🔗 Related Materials

- 📖 Reading: [06 — Dashboard Design & UI/UX](../readings/06_dashboard_design_ui_ux.md)
- 📖 Reading: [05 — Layouts, Containers & Page Structure](../readings/05_layouts_and_containers.md)
- 📓 Notebook: [05 — Layouts & Containers](05_layouts_and_containers.ipynb)
- ✏️ Exercise: [06 — Dashboard Builder](../exercises/06_dashboard_builder.py)
- 🖥️ Demo: [06 — Dashboard Demo](../apps/06_dashboard_demo.py)
- 📝 Quiz: [03 — Layouts & UI/UX](../quizzes/03_layouts_uiux.md)